# Recurrent STDP hyperparameter experiment trend

This notebook reads `experiment_results.tsv`, orders experiments by experiment ID, and plots validation-loss progress from the baseline. Aborted experiments remain in the status table but are excluded from metric curves.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_colwidth', 120)

RESULTS_PATH = Path('experiment_results.tsv')
if not RESULTS_PATH.exists():
    raise FileNotFoundError(f'Run this notebook from the repository root: {RESULTS_PATH.resolve()}')

raw = pd.read_csv(RESULTS_PATH, sep='\t', na_values=['NA'])
raw['experiment_no'] = raw['experiment'].str.extract(r'(\d+)', expand=False).astype(int)
raw = raw.sort_values('experiment_no').reset_index(drop=True)

completed = raw.loc[raw['status'].eq('ok')].copy()
completed['best_val'] = pd.to_numeric(completed['best_val'])
completed['test_loss'] = pd.to_numeric(completed['test_loss'])

baseline = completed.loc[completed['experiment'].eq('e01'), 'best_val'].iloc[0]
completed['improvement_vs_baseline'] = baseline - completed['best_val']
completed['running_best_val'] = completed['best_val'].cummin()
completed['running_best_improvement'] = baseline - completed['running_best_val']

best_idx = completed['best_val'].idxmin()
best = completed.loc[best_idx]

print(f'Baseline (e01): {baseline:.10f}')
print(f"Best ({best['experiment']}): {best['best_val']:.10f}")
print(f"Absolute improvement: {baseline - best['best_val']:.10f}")
print(f"Relative improvement: {(baseline - best['best_val']) / baseline * 100:.2f}%")

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10), sharex=True, constrained_layout=True)
x = completed['experiment_no'].to_numpy()

# Top: actual result for every completed experiment and the best known result at that point.
ax1.plot(x, completed['best_val'], color='#7f8c8d', marker='o', linewidth=1.4,
         alpha=0.75, label='Experiment val loss')
ax1.step(x, completed['running_best_val'], where='mid', color='#159957',
         linewidth=3, label='Running best val loss')
ax1.axhline(baseline, color='#c0392b', linestyle='--', linewidth=1.8,
            label=f'Baseline e01 ({baseline:.6f})')
ax1.scatter([best['experiment_no']], [best['best_val']], s=150, color='#f39c12',
            edgecolor='black', zorder=5, label=f"Best {best['experiment']} ({best['best_val']:.6f})")
ax1.annotate(f"{best['experiment']}\n{best['best_val']:.6f}",
             (best['experiment_no'], best['best_val']), xytext=(8, -30),
             textcoords='offset points', arrowprops={'arrowstyle': '->'})
ax1.set_ylabel('Validation loss (lower is better)')
ax1.set_title('Validation loss by experiment order')
ax1.legend(ncol=2)

# Bottom: positive values mean improvement over baseline.
colors = np.where(completed['improvement_vs_baseline'] >= 0, '#3498db', '#e74c3c')
ax2.bar(x, completed['improvement_vs_baseline'], color=colors, alpha=0.65,
        label='Experiment improvement')
ax2.step(x, completed['running_best_improvement'], where='mid', color='#159957',
         linewidth=3, label='Running best improvement')
ax2.axhline(0, color='black', linewidth=1)
ax2.set_ylabel('Baseline val loss - experiment val loss')
ax2.set_xlabel('Experiment order')
ax2.set_title('Improvement over baseline (higher is better)')
ax2.set_xticks(x)
ax2.set_xticklabels(completed['experiment'], rotation=45, ha='right')
ax2.legend()

plt.show()

In [ ]:
# Completed experiments in experiment order, including the exact change from baseline.
display_cols = [
    'experiment', 'best_val', 'running_best_val', 'improvement_vs_baseline',
    'test_loss', 'changes_from_baseline'
]
completed[display_cols].style.format({
    'best_val': '{:.10f}',
    'running_best_val': '{:.10f}',
    'improvement_vs_baseline': '{:+.10f}',
    'test_loss': '{:.10f}',
}).background_gradient(subset=['best_val'], cmap='RdYlGn_r')

In [ ]:
# Experiments that did not produce a complete 10-epoch result.
not_completed = raw.loc[~raw['status'].eq('ok'), [
    'experiment', 'status', 'changes_from_baseline'
]]
not_completed if not not_completed.empty else 'All experiments completed.'